# Choosing how many epochs to train the base model

### Imports

In [16]:
import sys
print(sys.version)

3.9.25 (main, Apr 17 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]


In [17]:
import os
import json

In [18]:
%ls

data/                     README.md
evaluation/               results/
master_auditor.ipynb      trainer/
master_experiment.ipynb   unlearn/
models/                   visualize_pretraining_results.ipynb
_old/                     visualize_results.ipynb
pretraining_models.ipynb  wandb/


In [19]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
# import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


### Set configs for the pretraining

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pretraining_config = {

    "description": "Testing pre-training on campus GPU - Resnet CIFAR10",
    
    "device": device,
    "model_class": "ResNet",
    "data": {
        "dataset": "CIFAR10",
        "num_classes": 10,
        "batch_size": 1024,
        "num_workers": 4,
        },

    "training": {
        "num_epochs": [1, 30, 50, 75, 100],
        "num_runs": 3,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_print_freq": 12,
        },
}

### Protocol for several runs

In [21]:
import wandb
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [22]:
import os
import json
import random
import glob
import torch
import wandb
import torch.nn as nn
import torch.optim as optim
from models.archs.utils import init_model
from data.dataloaders import load_dataloaders_for_experiment
from trainer.utils import init_folder_if_not_exists, training_regimen_lr_annealing
from trainer.val import validate
from models.archs.utils import init_model

def run_pretraining(config):

    print("-"*75)
    print("-"*13 + "  " + f"EVALUATING # OF EPOCHS FOR TRAINING {config['model_class']}" + "  " + "-"*13)
    print("-"*75 + "\n")

    # init wandb
    wandb.init(
      project="Verifying-Unlearning-2026",
      name=f"Pretraining Experiments - {config['model_class']}",
      config=config,
      reinit="finish_previous"
    )

    # Make experiment results folder if it doesnt already exist
    results_folder = init_folder_if_not_exists( f"results/pretraining/seed_{config['GRAND_SEED']}" )
    
    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # make model checkpoints folder for this seed if it doesn't exist yet
    checkpoints_folder = init_folder_if_not_exists( f"models/model_checkpoints/pretrained/seed_{config['GRAND_SEED']}" )


    for num_epochs in config["training"]["num_epochs"]:

        epoch_results_folder = init_folder_if_not_exists( os.path.join(results_folder, f"{config['data']['dataset']}_{config['model_class']}_{num_epochs}_epochs") )
        epoch_checkpoints_folder = init_folder_if_not_exists( os.path.join(checkpoints_folder, f"{config['data']['dataset']}_{config['model_class']}_{num_epochs}_epochs") )


        # get some data (does not change in between runs)
        train_loader, _, test_loader = load_dataloaders_for_experiment(
            name = config["data"]["dataset"],
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed=config["GRAND_SEED"], 
            class_to_replace=None, 
            percent_to_replace=None,
            val=False
            )

        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ---------------- TRAIN A BASE MODEL, FROM WHICH UNLEARNING BEGINS ----------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #


        print("-"*55)
        print("-"*13 + "  " + f"TESTING: {num_epochs} EPOCHS" + "  " + "-"*13)
        print("-"*55 + "\n")

        for i in range(1, config["training"]["num_runs"] + 1):

            # init model, opt, criterion, and scheduler
            empty_model = init_model(model_class = config["model_class"], num_classes=config["data"]["num_classes"]).to(config["device"])
            criterion = nn.CrossEntropyLoss()
            opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                opt, 
                T_max=num_epochs, 
                eta_min=1e-6
            )

            # train (wandb logging underneath, dont need to re-log training accuracy)
            base_model_path = os.path.join(epoch_checkpoints_folder, f"{config['model_class']}_{i}.pth")
            trained_model, opt, scheduler, _, _, _, _ = training_regimen_lr_annealing(
                empty_model, 
                train_loader,
                opt, 
                criterion, 
                scheduler, 
                device = config["device"], 
                num_epochs=num_epochs, 
                model_path = base_model_path,
                print_freq = config["training"]["batch_print_freq"])

            # evaluate trained model on some test
            print(f"Evaluating model trained for {num_epochs} epochs on test set...\n") 
            train_out = validate(
                train_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )
            
            test_out = validate(
                test_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )

            print(f"Train accuracy: {train_out['avg_acc']:.4f}\n")
            print(f"Test accuracy: {test_out['avg_acc']:.4f}\n")
            
            results = {
                "num_epochs": num_epochs,
                "train_acc": train_out["avg_acc"],
                "test_acc": test_out["avg_acc"]
                }

            # save results
            with open(os.path.join(epoch_results_folder, f"results_{i}.json"), "w") as f:
                json.dump(results, f, indent=4)

            # Save checkpoint
            print(f"Saving base model to {base_model_path}...")
            torch.save(trained_model.state_dict(), base_model_path)
                    
    wandb.finish()

    print("-"*70)
    print("-"*19 + "  " + f"FINISHED PRETRAINING" + "  " + "-"*19)
    print("-"*70 + "\n")


### Check metrics on unlearned models

In [ ]:
# MAKE A RANDOM SEED
pretraining_config["GRAND_SEED"] = 4
# DO EXP
run_pretraining(config = pretraining_config)

---------------------------------------------------------------------------
-------------  EVALUATING # OF EPOCHS FOR TRAINING ResNet  -------------
---------------------------------------------------------------------------



wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▁█████████████████████████████
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
learning_rate,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
time (batch),▁▁▇▁▁▁▁█▂▂▇▁▇▁▇▁▁▁▁▁▁▁▁▇▇▁▁▁▁▇▁▁▁▁▇▁▇▁▁▁
train_acc (batch),▁▃▃▄▄▅▅▆▆▆▆▇▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████
train_acc (full),▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████████
train_entropy (batch),█▆▅▅▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train_loss (batch),██▇▆▅▄▃▃▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁
+4,...


results/pretraining/seed_4/CIFAR10_ResNet_1_epochs doesn't exist - creating it...

models/model_checkpoints/pretrained/seed_4/CIFAR10_ResNet_1_epochs doesn't exist - creating it...

========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


-------------------------------------------------------
-------------  TESTING: 1 EPOCHS  -------------
-------------------------------------------------------

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/49]	Loss 1.6923 (2.0418)	Accuracy 36.133 (25.529)	Entropy 1.6262 (1.8312)	M-Entropy 1.6129 (1.9843)	Time 3.34
Epoch: [1][23/49]	Loss 1.4918 (1.8336)	Accuracy 45.020 (32.353)	Entropy 1.5223 (1.7293)	M-Entropy 1.4181 (1.7616)	Time 2.51
Epoch: [1][35/49]	Loss 1.3577 (1.7088)	Accuracy 49.023 (36.936)